# 10. Preprocesamiento y Pipeline (2.9)

Todo el preprocesamiento vive dentro de un `Pipeline` de scikit-learn
{cite}`pedregosa_2011` y se ajusta **solo con el entrenamiento** (en cada pliegue de la
validación cruzada, solo con la parte de entrenamiento del pliegue). Cada decisión se
vincula con el hallazgo del EDA que la motiva.

In [1]:
from pathlib import Path
import re, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

project_root = Path.cwd().resolve()
while not (project_root / "_config.yml").exists():
    if project_root.parent == project_root:
        raise FileNotFoundError("No se encontró _config.yml. Ejecuta el notebook dentro de la carpeta del libro.")
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

for _folder in ("tables", "figures"):          # se crean si no existen
    (project_root / _folder).mkdir(exist_ok=True)

from src.project_utils import *

configure_plots()
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 70)
TABLES = project_root / "tables"

def md_out(text):
    display(Markdown(text))

from sklearn.model_selection import cross_validate
from sklearn import set_config
warnings.filterwarnings("ignore", message="Found unknown categories")
decision = pd.read_csv(TABLES / "decision_poblacion.csv").iloc[0]
train = load_train(include_canceled=not bool(decision["excluir_cancelados"]))
y = train[TARGET]
folds = temporal_folds(train[DATE_COL])
md_out(f"Población de modelado (decisión del cap. 9): **{es(len(train), 0)}** líneas de entrenamiento "
       f"({'sin' if decision['excluir_cancelados'] else 'con'} pedidos cancelados/fraude); tasa de tardías {es(y.mean(), 1, pct=True)}.")

Población de modelado (decisión del cap. 9): **137.043** líneas de entrenamiento (sin pedidos cancelados/fraude); tasa de tardías 57,3 %.

## Decisiones y hallazgo que las motiva

In [2]:
decisiones = pd.DataFrame([
    ("Excluir variables posteriores al pedido", "days_shipping_real, delivery_status, order_status, shipping_date", "Cap. 9: AUC 0,97–1 con fuga; no disponibles al predecir"),
    ("Excluir pedidos cancelados/fraude", "población", "Cap. 2 y 9: etiqueta 0 por construcción; vuelven proxy al medio de pago"),
    ("Excluir identificadores y datos personales", "ids, nombre, dirección, correo...", "Cap. 2 y 9: sin significado predictivo; riesgo ético"),
    ("Una columna por par duplicado", "sales_per_customer, order_profit_per_order, product_price...", "Cap. 2: copias exactas; cap. 5: VIF infinito"),
    ("shipping_mode en lugar de days_shipping_scheduled", "shipping_mode", "Cap. 2: relación 1 a 1; la categórica no impone linealidad entre 0, 1, 2 y 4 días"),
    ("Omitir precio y descuento absolutos", "order_item_product_price, order_item_discount", "Cap. 5: VIF > 10 (ventas = precio × cantidad)"),
    ("log1p de ventas", "sales", "Cap. 4: asimetría positiva y valores altos"),
    ("Escalado estándar de numéricas", "todas las numéricas", "Coeficientes comparables; regularización L2 equilibrada"),
    ("Interacción modo × jornada (AM/PM)", "mode_x_daypart", "Cap. 7 y 9: hora de corte del mediodía solo en Same Day"),
    ("Seno/coseno de hora, día y mes", "order_hour/dow/month", "Cap. 7: variables cíclicas (23 h cerca de 0 h)"),
    ("Carga de los 7 días previos", "orders_prev_7d", "Cap. 7: ventana solo con el pasado (cerrada a la izquierda)"),
    ("One-hot con agrupación de raras (< 0,5 %)", "categóricas", "Cap. 4: alta cardinalidad en país y categorías raras"),
    ("Imputación dentro del Pipeline", "mediana / «Missing»", "Cap. 2: sin faltantes en las candidatas; se deja por robustez ante datos nuevos"),
    ("Conservar atípicos", "ganancia, ventas", "Cap. 2, 4 y 6: son ventas reales con pérdida o productos caros; log y escalado reducen su influencia"),
    ("Destino y coordenadas: decidir con CV temporal", "market, order_region, order_country, lat/lon", "Cap. 7 y 8: sin señal y confundidos con el periodo"),
], columns=["decisión", "variables", "hallazgo del EDA"])
decisiones.to_csv(TABLES / "decisiones_preprocesamiento.csv", index=False)
decisiones

,decisión,variables,hallazgo del EDA
0,Excluir variables posteriores al pedido,"days_shipping_real, delivery_status, order_status, shipping_date","Cap. 9: AUC 0,97–1 con fuga; no disponibles al predecir"
1,Excluir pedidos cancelados/fraude,población,Cap. 2 y 9: etiqueta 0 por construcción; vuelven proxy al medio de...
2,Excluir identificadores y datos personales,"ids, nombre, dirección, correo...",Cap. 2 y 9: sin significado predictivo; riesgo ético
3,Una columna por par duplicado,"sales_per_customer, order_profit_per_order, product_price...",Cap. 2: copias exactas; cap. 5: VIF infinito
4,shipping_mode en lugar de days_shipping_scheduled,shipping_mode,Cap. 2: relación 1 a 1; la categórica no impone linealidad entre 0...
5,Omitir precio y descuento absolutos,"order_item_product_price, order_item_discount",Cap. 5: VIF > 10 (ventas = precio × cantidad)
6,log1p de ventas,sales,Cap. 4: asimetría positiva y valores altos
7,Escalado estándar de numéricas,todas las numéricas,Coeficientes comparables; regularización L2 equilibrada
8,Interacción modo × jornada (AM/PM),mode_x_daypart,Cap. 7 y 9: hora de corte del mediodía solo en Same Day
9,"Seno/coseno de hora, día y mes",order_hour/dow/month,Cap. 7: variables cíclicas (23 h cerca de 0 h)


## Selección del conjunto de variables con validación cruzada temporal

Se comparan conjuntos anidados con la **misma** regresión logística y los mismos pliegues
temporales con *gap* (solo entrenamiento). La regla, fijada de antemano: elegir el conjunto
más simple cuyo ROC-AUC medio esté a menos de una desviación estándar del mejor.

In [3]:
CICLICAS = ["order_hour_sin", "order_hour_cos", "order_dow_sin", "order_dow_cos", "order_month_sin", "order_month_cos"]
BASE_NUM_LIN = ["order_item_quantity", "order_item_discount_rate", "order_item_profit_ratio", "items_in_order", "orders_prev_7d", "order_weekend"] + CICLICAS
BASE_NUM_LOG = ["sales"]
BASE_CAT = ["shipping_mode", "payment_type", "customer_segment", "department_name", "category_name"]
CAT_INT = ["mode_x_daypart"] + BASE_CAT[1:]          # la interacción reemplaza al modo solo
DEST = ["market", "order_region", "order_country"]
conjuntos = {
    "A · solo modo de envío": ([], [], ["shipping_mode"]),
    "B · modo + hora (aditivo, seno/coseno)": (["order_hour_sin", "order_hour_cos"], [], ["shipping_mode"]),
    "C · modo × jornada (interacción)": ([], [], ["mode_x_daypart"]),
    "D · pedido completo + interacción (sin destino)": (BASE_NUM_LIN, BASE_NUM_LOG, CAT_INT),
    "E · D + destino (mercado, región, país)": (BASE_NUM_LIN, BASE_NUM_LOG, CAT_INT + DEST),
    "F · E + coordenadas": (BASE_NUM_LIN + [LAT, LON], BASE_NUM_LOG, CAT_INT + DEST),
}
filas = []
for nombre, (nl, nlog, cat) in conjuntos.items():
    pipe = make_model_pipeline(nl, nlog, cat)
    cv = cross_validate(pipe, train[nl + nlog + cat], y, cv=folds, scoring=["roc_auc", "accuracy"], n_jobs=1)
    filas.append({"conjunto": nombre, "n_variables": len(nl + nlog + cat), "ROC-AUC media": cv["test_roc_auc"].mean(),
                  "ROC-AUC desv.": cv["test_roc_auc"].std(), "Accuracy media": cv["test_accuracy"].mean()})
comp = pd.DataFrame(filas)
comp.to_csv(TABLES / "seleccion_conjuntos.csv", index=False)
import json
(TABLES / "conjuntos_variables.json").write_text(json.dumps({k: {"num_lin": a, "num_log": b, "cat": c} for k, (a, b, c) in conjuntos.items()},
                                                           ensure_ascii=False, indent=1), encoding="utf-8")
display(comp.round(4))
mejor = comp.loc[comp["ROC-AUC media"].idxmax()]
elegible = comp[comp["ROC-AUC media"] >= mejor["ROC-AUC media"] - mejor["ROC-AUC desv."]]
elegido = elegible.sort_values("n_variables").iloc[0]["conjunto"]
md_out(f"Mejor ROC-AUC medio: **{mejor['conjunto']}** ({es(mejor['ROC-AUC media'], 4)} ± {es(mejor['ROC-AUC desv.'], 4)}). "
       f"Conjunto más simple dentro de una desviación estándar: **{elegido}**.")

,conjunto,n_variables,ROC-AUC media,ROC-AUC desv.,Accuracy media
0,A · solo modo de envío,1,0.7401,0.0023,0.6963
1,"B · modo + hora (aditivo, seno/coseno)",3,0.7458,0.0032,0.7089
2,C · modo × jornada (interacción),1,0.7740,0.0033,0.7216
3,D · pedido completo + interacción (sin destino),18,0.7722,0.0029,0.7216
4,"E · D + destino (mercado, región, país)",21,0.7723,0.0037,0.7211
5,F · E + coordenadas,23,0.7724,0.0038,0.7211


Mejor ROC-AUC medio: **C · modo × jornada (interacción)** (0,7740 ± 0,0033). Conjunto más simple dentro de una desviación estándar: **C · modo × jornada (interacción)**.

**Interpretación.** Añadir la hora de forma **aditiva** (B) mejora poco sobre el modo solo
(A), porque supone que la hora afecta igual a todos los modos. La **interacción modo × jornada**
(C) captura la hora de corte del mediodía en Same Day y da el salto grande: alcanza el techo
sin fuga del capítulo 9 (≈ 0,77). Las variables de producto, pago, segmento, calendario y
carga (D) **no añaden nada**, y el **destino** y las **coordenadas** (E, F) tampoco (diferencias
menores a 0,001, muy por debajo de la variación entre pliegues). Por la regla fijada
de antemano, el modelo base usa el conjunto **C: una sola variable categórica de 8 niveles
(modo × jornada)**. El conjunto D se reporta en el capítulo 11 como **análisis de
sensibilidad**, para mostrar que añadir variables no cambia la conclusión.

In [4]:
nl, nlog, cat = conjuntos[elegido]
modelo_cols = {"num_lin": nl, "num_log": nlog, "cat": cat, "conjunto": elegido}
(TABLES / "modelo_elegido.json").write_text(json.dumps(modelo_cols, ensure_ascii=False, indent=1), encoding="utf-8")
pd.DataFrame([{"tipo": k, "columnas": ", ".join(v) if isinstance(v, list) else v} for k, v in modelo_cols.items()]).to_csv(TABLES / "columnas_modelo.csv", index=False)
pipe = make_model_pipeline(nl, nlog, cat)
set_config(display="diagram")
display(pipe)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  Pipeline(steps=[('imp',
                                                                   SimpleImputer(fill_value='Missing',
                                                                                 keep_empty_features=True,
                                                                                 strategy='constant')),
                                                                  ('oh',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='infrequent_if_exist',
                                                                                 min_frequency=0.005))]),
                                                  ['mode_x_daypart'])])),
                ('clf', LogisticRegression(max_iter=3000, random_state=42))])

## Comprobación: el Pipeline se ajusta solo con entrenamiento

In [5]:
pipe.fit(train[nl + nlog + cat], y)
prep = pipe.named_steps["prep"]
nombres = prep.get_feature_names_out()
md_out(f"Columnas tras el preprocesamiento: **{len(nombres)}**.")
info = []
for name, trans, cols_ in prep.transformers_:
    if name == "remainder":
        continue
    if name in ("num", "numlog"):
        sc = trans.named_steps["sc"]
        for c, m, s in zip(cols_, sc.mean_, sc.scale_):
            info.append({"bloque": name, "variable": c, "media_ajustada": m, "escala_ajustada": s, "categorías": ""})
    else:
        oh = trans.named_steps["oh"]
        for c, cats_, inf in zip(cols_, oh.categories_, oh.infrequent_categories_):
            info.append({"bloque": name, "variable": c, "media_ajustada": np.nan, "escala_ajustada": np.nan,
                         "categorías": f"{len(cats_)} categorías; agrupadas como infrecuentes: {0 if inf is None else len(inf)}"})
info = pd.DataFrame(info)
info.to_csv(TABLES / "parametros_preprocesamiento.csv", index=False)
display(info.round(4))
Xt = prep.transform(train[nl + nlog + cat])
md_out(f"Matriz transformada: {es(Xt.shape[0], 0)} × {Xt.shape[1]}. Media de las numéricas transformadas ≈ 0 y desviación ≈ 1 en entrenamiento "
       "(por construcción). En la prueba se aplican **estos mismos** parámetros, sin reajustar.")

Columnas tras el preprocesamiento: **7**.

,bloque,variable,media_ajustada,escala_ajustada,categorías
0,cat,mode_x_daypart,NaN,NaN,8 categorías; agrupadas como infrecuentes: 0


Matriz transformada: 137.043 × 7. Media de las numéricas transformadas ≈ 0 y desviación ≈ 1 en entrenamiento (por construcción). En la prueba se aplican **estos mismos** parámetros, sin reajustar.

**Resumen.** El Pipeline imputa (por robustez), transforma con `log1p` las ventas, escala las
numéricas, codifica las categóricas con *one-hot* (referencia = primera categoría; raras
agrupadas) y ajusta la regresión logística. Al estar todo dentro del Pipeline, la validación
cruzada del capítulo 11 reajusta cada paso dentro de cada pliegue: **no hay fuga por
preprocesamiento**.